In [1]:
import numpy as np

# InvertedPendulum Timesteps to first solve the environment

In [2]:
# This is for DDPG, RPI and TD3.
def get_data_for_timesteps_table(folder_path, reward_threshold, seeds=list(range(25)), verbose=False):
    time_steps_first_reward_threshold = []

    for seed in seeds:
        try:
            total_rewards_seed = np.load(folder_path+f"/{seed}/total_rew_vals.npy")
            time_steps_seed = np.load(folder_path+f"/{seed}/time_steps.npy")
            tot_rewards_mean = np.array([np.mean(x) for x in total_rewards_seed])    
            first_reward_threshold_index = np.where(tot_rewards_mean >= reward_threshold)[0][0]
            time_steps_first_reward_threshold.append(time_steps_seed[first_reward_threshold_index])
        except:
            if verbose:
                print(f"Threshold did not reach in folder: {folder_path} for seed: {seed}")

    mean_timesteps = np.mean(time_steps_first_reward_threshold)
    std_timesteps = np.std(time_steps_first_reward_threshold)
    num_seeds_success = len(time_steps_first_reward_threshold)

    return mean_timesteps, std_timesteps, num_seeds_success


def get_data_for_timesteps_table_ppo(folder_path, reward_threshold, seeds=list(range(25)), n_env=1, verbose=False):
    time_steps_first_reward_threshold = []

    for seed in seeds:
        try:
            # TODO: Remove allow_pickle=True after the new code is run.
            total_rewards_seed = np.squeeze(np.load(folder_path+f"/seed_{seed}/mc_tot_returns.npy", allow_pickle=True))
            time_steps_seed = np.array([x[0] for x in np.load(folder_path+f"/seed_{seed}/time_steps_of_eval.npy", allow_pickle=True)])
            tot_rewards_mean = np.array([np.mean(x) for x in total_rewards_seed])    
            first_reward_threshold_index = np.where(tot_rewards_mean >= reward_threshold)[0][0]
            time_steps_first_reward_threshold.append(time_steps_seed[first_reward_threshold_index] * n_env)
        except:
            if verbose:
                print(f"Threshold did not reach in folder: {folder_path} for seed: {seed}")

    mean_timesteps = np.mean(time_steps_first_reward_threshold)
    std_timesteps = np.std(time_steps_first_reward_threshold)
    num_seeds_success = len(time_steps_first_reward_threshold)

    return mean_timesteps, std_timesteps, num_seeds_success

## Network Architecture change

In [3]:
reward_threshold = 1000
num_seeds = 10
seeds = list(range(num_seeds))
algos = ["DDPG_RPI", "DDPG", "TD3", "PPO"]
num_neurons = ["32-32", "64-64", "128-128", "256-256", "400-300", "512-512"]

data = {}
for num_neuron in num_neurons:
    data[num_neuron] = {}
    for algo in algos:
        try:
            if algo == "PPO":
                folder_path = f"Net-Arch-Exp_PPO/PPO_Runs/results/InvertedPendulum-v5/PPO/arch_{num_neuron}"
                mean_timesteps, std_timesteps, num_seeds_success = get_data_for_timesteps_table_ppo(folder_path, reward_threshold, seeds) # n_env=1 in InvertedPendulum
            else:
                folder_path = f"results/net-arch/InvertedPendulum-v5/{algo}/arch_{num_neuron}"
                mean_timesteps, std_timesteps, num_seeds_success = get_data_for_timesteps_table(folder_path, reward_threshold, seeds)
            # print("folder_path:", folder_path)
            
            data[num_neuron][algo] = {
                "mean_timesteps": mean_timesteps,
                "std_timesteps": std_timesteps,
                "num_seeds_success": num_seeds_success
            }
        except:
            pass

print(data)

{'32-32': {'DDPG_RPI': {'mean_timesteps': np.float64(18800.0), 'std_timesteps': np.float64(3370.4599092705434), 'num_seeds_success': 10}, 'DDPG': {'mean_timesteps': np.float64(25700.0), 'std_timesteps': np.float64(9349.331526906082), 'num_seeds_success': 10}, 'TD3': {'mean_timesteps': np.float64(42300.0), 'std_timesteps': np.float64(9066.97303403953), 'num_seeds_success': 10}, 'PPO': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, '64-64': {'DDPG_RPI': {'mean_timesteps': np.float64(15900.0), 'std_timesteps': np.float64(3562.302626111375), 'num_seeds_success': 10}, 'DDPG': {'mean_timesteps': np.float64(18800.0), 'std_timesteps': np.float64(13181.805642627265), 'num_seeds_success': 10}, 'TD3': {'mean_timesteps': np.float64(35900.0), 'std_timesteps': np.float64(7354.590403278758), 'num_seeds_success': 10}, 'PPO': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, '128-128': {'DDPG_RPI': {'mean_tim

/Users/eshwar/miniforge3/envs/rpi-rl-env/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/Users/eshwar/miniforge3/envs/rpi-rl-env/lib/python3.12/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/eshwar/miniforge3/envs/rpi-rl-env/lib/python3.12/site-packages/numpy/_core/_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/eshwar/miniforge3/envs/rpi-rl-env/lib/python3.12/site-packages/numpy/_core/_methods.py:181: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/Users/eshwar/miniforge3/envs/rpi-rl-env/lib/python3.12/site-packages/numpy/_core/_methods.py:215: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [4]:
print("Reward threshold: ", reward_threshold)

# 1. Define column widths
W_ALGO = 12
W_NEURON = 10
W_SEEDS = 18
W_MEAN = 18
W_STD = 16

# 2. Print the header
# < : left-align, > : right-align
header = (
    f"{'Algo':<{W_ALGO}}"
    f"{'Neurons':>{W_NEURON}}"
    f"{f'Successful Seeds ({num_seeds})':>{W_SEEDS}}"
    f"{'Mean Timesteps':>{W_MEAN}}"
    f"{'Std Timesteps':>{W_STD}}"
)
print(header)
print("-" * len(header))

# 3. Iterate and print the data
found_data = False
for num_neuron in num_neurons:
    if num_neuron in data:
        for algo in algos:
            if algo in data[num_neuron]:
                found_data = True
                metrics = data[num_neuron][algo]
                
                # Format and print the row
                # :.2f formats floats to 2 decimal places
                print(
                    f"{algo:<{W_ALGO}}"
                    f"{num_neuron:>{W_NEURON}}"
                    f"{metrics['num_seeds_success']:>{W_SEEDS}}"
                    f"{metrics['mean_timesteps']:>{W_MEAN}.2f}"
                    f"{metrics['std_timesteps']:>{W_STD}.2f}"
                )
    print("-" * len(header))

if not found_data:
    print("No data found to display.")

Reward threshold:  1000
Algo           NeuronsSuccessful Seeds (10)    Mean Timesteps   Std Timesteps
-----------------------------------------------------------------------------
DDPG_RPI         32-32                10          18800.00         3370.46
DDPG             32-32                10          25700.00         9349.33
TD3              32-32                10          42300.00         9066.97
PPO              32-32                 0               nan             nan
-----------------------------------------------------------------------------
DDPG_RPI         64-64                10          15900.00         3562.30
DDPG             64-64                10          18800.00        13181.81
TD3              64-64                10          35900.00         7354.59
PPO              64-64                 0               nan             nan
-----------------------------------------------------------------------------
DDPG_RPI       128-128                10          11000.00      

# Env Configs

In [5]:
reward_threshold = 1000
num_seeds = 10
seeds = list(range(num_seeds))
algos = ["DDPG_RPI"]
env_configs = ["g_-9.81_cm_10.47197551_pm_5.01859164", 
               "g_-4.905_cm_10.47197551_pm_5.01859164", "g_-19.62_cm_10.47197551_pm_5.01859164",
               "g_-9.81_cm_5.235987755_pm_5.01859164", "g_-9.81_cm_20.94395102_pm_5.01859164",
               "g_-9.81_cm_10.47197551_pm_2.50929582", "g_-9.81_cm_10.47197551_pm_10.03718328"]

data = {}
for env_config in env_configs:
    data[env_config] = {}
    for algo in algos:
        try:
            folder_path = f"results/env-config/InvertedPendulum-v5/{algo}/{env_config}"
            mean_timesteps, std_timesteps, num_seeds_success = get_data_for_timesteps_table(folder_path, reward_threshold, seeds)
            data[env_config][algo] = {
                "mean_timesteps": mean_timesteps,
                "std_timesteps": std_timesteps,
                "num_seeds_success": num_seeds_success
            }
        except:
            pass

print(data)

{'g_-9.81_cm_10.47197551_pm_5.01859164': {'DDPG_RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 'g_-4.905_cm_10.47197551_pm_5.01859164': {'DDPG_RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 'g_-19.62_cm_10.47197551_pm_5.01859164': {'DDPG_RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 'g_-9.81_cm_5.235987755_pm_5.01859164': {'DDPG_RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 'g_-9.81_cm_20.94395102_pm_5.01859164': {'DDPG_RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 'g_-9.81_cm_10.47197551_pm_2.50929582': {'DDPG_RPI': {'mean_timesteps': np.float64(nan), 'std_timesteps': np.float64(nan), 'num_seeds_success': 0}}, 'g_-9.81_cm_10.47197551_pm_10.03718328': {'DDPG_RPI': {'mean_timesteps': np.float64(nan), 'std_timestep

In [6]:
print("Reward threshold: ", reward_threshold)

# 1. Define column widths
W_ALGO = 12
W_ENV_CONFIG = 40
W_SEEDS = 18
W_MEAN = 18
W_STD = 16

# 2. Print the header
# < : left-align, > : right-align
header = (
    f"{'Algo':<{W_ALGO}}"
    f"{'Env_Config ':>{W_ENV_CONFIG}}"
    f"{f'Successful Seeds ({num_seeds})':>{W_SEEDS}}"
    f"{'Mean Timesteps':>{W_MEAN}}"
    f"{'Std Timesteps':>{W_STD}}"
)
print(header)
print("-" * len(header))

# 3. Iterate and print the data
found_data = False
for env_config in env_configs:
    if env_config in data:
        for algo in algos:
            if algo in data[env_config]:
                found_data = True
                metrics = data[env_config][algo]
                
                # Format and print the row
                # :.2f formats floats to 2 decimal places
                print(
                    f"{algo:<{W_ALGO}}"
                    f"{env_config:>{W_ENV_CONFIG}}"
                    f"{metrics['num_seeds_success']:>{W_SEEDS}}"
                    f"{metrics['mean_timesteps']:>{W_MEAN}.2f}"
                    f"{metrics['std_timesteps']:>{W_STD}.2f}"
                )
    print("-" * len(header))

if not found_data:
    print("No data found to display.")

Reward threshold:  1000
Algo                                     Env_Config Successful Seeds (10)    Mean Timesteps   Std Timesteps
-----------------------------------------------------------------------------------------------------------
DDPG_RPI        g_-9.81_cm_10.47197551_pm_5.01859164                 0               nan             nan
-----------------------------------------------------------------------------------------------------------
DDPG_RPI       g_-4.905_cm_10.47197551_pm_5.01859164                 0               nan             nan
-----------------------------------------------------------------------------------------------------------
DDPG_RPI       g_-19.62_cm_10.47197551_pm_5.01859164                 0               nan             nan
-----------------------------------------------------------------------------------------------------------
DDPG_RPI        g_-9.81_cm_5.235987755_pm_5.01859164                 0               nan             nan
----------------

# Penalty Functions

In [7]:
reward_threshold = 1000
num_seeds = 10
seeds = list(range(num_seeds))
algos = ["DDPG_RPI"]
penalty_functions = ["penalty_relu_False","penalty_cubic_True", "penalty_cubic_False", "penalty_quadratic_True", "penalty_quadratic_False"]

data = {}
for penalty_function in penalty_functions:
    data[penalty_function] = {}
    for algo in algos:
        try:
            folder_path = f"results/penalty-fn/InvertedPendulum-v5/{algo}/{penalty_function}"
            mean_timesteps, std_timesteps, num_seeds_success = get_data_for_timesteps_table(folder_path, reward_threshold, seeds)
            data[penalty_function][algo] = {
                "mean_timesteps": mean_timesteps,
                "std_timesteps": std_timesteps,
                "num_seeds_success": num_seeds_success
            }
        except:
            pass

print(data)

{'penalty_relu_False': {'DDPG_RPI': {'mean_timesteps': np.float64(8700.0), 'std_timesteps': np.float64(2147.091055358389), 'num_seeds_success': 10}}, 'penalty_cubic_True': {'DDPG_RPI': {'mean_timesteps': np.float64(16500.0), 'std_timesteps': np.float64(3612.4783736376885), 'num_seeds_success': 10}}, 'penalty_cubic_False': {'DDPG_RPI': {'mean_timesteps': np.float64(12222.222222222223), 'std_timesteps': np.float64(1474.0554623801777), 'num_seeds_success': 9}}, 'penalty_quadratic_True': {'DDPG_RPI': {'mean_timesteps': np.float64(13400.0), 'std_timesteps': np.float64(2576.8197453450252), 'num_seeds_success': 10}}, 'penalty_quadratic_False': {'DDPG_RPI': {'mean_timesteps': np.float64(12400.0), 'std_timesteps': np.float64(3498.57113690718), 'num_seeds_success': 10}}}


In [8]:
print("Reward threshold: ", reward_threshold)

# 1. Define column widths
W_ALGO = 12
W_PENALTY_FUNCTION = 30
W_SEEDS = 18
W_MEAN = 18
W_STD = 16

# 2. Print the header
# < : left-align, > : right-align
header = (
    f"{'Algo':<{W_ALGO}}"
    f"{'Penalty function ':>{W_PENALTY_FUNCTION}}"
    f"{f'Successful Seeds ({num_seeds})':>{W_SEEDS}}"
    f"{'Mean Timesteps':>{W_MEAN}}"
    f"{'Std Timesteps':>{W_STD}}"
)
print(header)
print("-" * len(header))

# 3. Iterate and print the data
found_data = False
for penalty_function in penalty_functions:
    if penalty_function in data:
        for algo in algos:
            if algo in data[penalty_function]:
                found_data = True
                metrics = data[penalty_function][algo]
                
                # Format and print the row
                # :.2f formats floats to 2 decimal places
                print(
                    f"{algo:<{W_ALGO}}"
                    f"{penalty_function:>{W_PENALTY_FUNCTION}}"
                    f"{metrics['num_seeds_success']:>{W_SEEDS}}"
                    f"{metrics['mean_timesteps']:>{W_MEAN}.2f}"
                    f"{metrics['std_timesteps']:>{W_STD}.2f}"
                )
    print("-" * len(header))

if not found_data:
    print("No data found to display.")

Reward threshold:  1000
Algo                     Penalty function Successful Seeds (10)    Mean Timesteps   Std Timesteps
-------------------------------------------------------------------------------------------------
DDPG_RPI                penalty_relu_False                10           8700.00         2147.09
-------------------------------------------------------------------------------------------------
DDPG_RPI                penalty_cubic_True                10          16500.00         3612.48
-------------------------------------------------------------------------------------------------
DDPG_RPI               penalty_cubic_False                 9          12222.22         1474.06
-------------------------------------------------------------------------------------------------
DDPG_RPI            penalty_quadratic_True                10          13400.00         2576.82
-------------------------------------------------------------------------------------------------
DDPG_RPI